# Cotton Disease Classification

This notebook trains and evaluates CNN and Transformer models on the cotton disease dataset,
then fuses the two best-performing backbones using multiple feature fusion strategies.

The workflow has two main parts:

**Part 1 - Individual Models**
Train MobileNetV2, ShuffleNetV2, EfficientNet-B3, DeiT-Tiny, and Swin-Tiny using
5-fold cross-validation. The best checkpoint from each model is saved to disk.

**Part 2 - Feature Fusion**
Load the frozen DeiT-Tiny and ShuffleNetV2 backbones and train lightweight fusion heads
using five strategies: concat, weighted sum, bilinear, cross-attention, and gated.

Dataset classes: bacterial_blight, curl_virus, fusarium_wilt, healthy

## 0. Imports and Setup

In [ ]:
import subprocess
import sys


def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


try:
    import timm
except ImportError:
    pip_install("timm")
    import timm

import copy
import math
import os
import random
import time

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch import optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models
from torchvision.datasets import ImageFolder

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

## 1. Paths and Dataset Configuration

In [ ]:
# Paths - update these to match your directory layout
TRAIN_PATH = "cotton_split/Train"
TEST_PATH = "cotton_split/Test"

# Output directories
OUTPUT_DIR_INDIVIDUALS = "cotton_individual_model_results"
OUTPUT_DIR_FUSION = "cotton_fusion_results"
SAVED_MODELS_DIR = "cotton_saved_models"

for d in [
    OUTPUT_DIR_INDIVIDUALS,
    OUTPUT_DIR_FUSION,
    SAVED_MODELS_DIR,
    os.path.join(OUTPUT_DIR_INDIVIDUALS, "confusion_matrices"),
    os.path.join(OUTPUT_DIR_FUSION, "confusion_matrices"),
    os.path.join(OUTPUT_DIR_FUSION, "visualizations"),
]:
    os.makedirs(d, exist_ok=True)

# Dataset info
CLASS_NAMES = ["bacterial_blight", "curl_virus", "fusarium_wilt", "healthy"]
NUM_CLASSES = 4

# Checkpoint names for the two backbones that will be used in fusion
# These are filled automatically after Part 1 completes
DEIT_CKPT = None
SHUFFLENET_CKPT = None

# Load training set as the primary dataset for cross-validation
full_dataset = ImageFolder(root=TRAIN_PATH, transform=None)
test_dataset_raw = ImageFolder(root=TEST_PATH, transform=None)

print(f"Classes   : {full_dataset.classes}")
print(f"Train imgs: {len(full_dataset)}")
print(f"Test imgs : {len(test_dataset_raw)}")
print("Class distribution (train):")
train_labels = np.array([lbl for _, lbl in full_dataset.samples])
for name, idx in full_dataset.class_to_idx.items():
    print(f"  {name}: {(train_labels == idx).sum()}")

---
# Part 1 - Individual Model Training

Each model is trained with 5-fold stratified cross-validation.
The fold with the highest validation accuracy is saved as the model checkpoint.

## 2. Preprocessing and Data Utilities

In [ ]:
class SobelEdgeDetectionTransform:
    """Convert an RGB PIL image to a 3-channel Sobel edge map."""

    def __call__(self, img):
        arr = np.array(img)
        sobel_x = cv2.Sobel(arr, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(arr, cv2.CV_64F, 0, 1, ksize=3)
        sobel = np.hypot(sobel_x, sobel_y)
        max_value = max(float(sobel.max()), 1.0)
        sobel = np.uint8(255 * sobel / max_value)
        return Image.fromarray(sobel)


def build_transforms(use_sobel=False, is_transformer=False):
    """Return (train_transform, val_transform) for a given model configuration."""
    pre_ops = [SobelEdgeDetectionTransform()] if use_sobel else []

    train_aug = [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.70, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
        transforms.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
    ]

    if is_transformer:
        # Mild blur helps transformers generalise on small datasets
        train_aug.insert(
            1,
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=3)], p=0.15
            ),
        )

    train_transform = transforms.Compose(pre_ops + train_aug)
    val_transform = transforms.Compose(
        pre_ops
        + [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            ),
        ]
    )
    return train_transform, val_transform


class TransformSubset(Dataset):
    """Wrap a subset of an ImageFolder with its own transform.

    This avoids the transform-leakage bug that occurs when the shared
    ImageFolder transform is mutated during cross-validation.
    """

    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [
            base_dataset.samples[idx][1] for idx in self.indices
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample_idx = self.indices[idx]
        path, label = self.base_dataset.samples[sample_idx]
        image = self.base_dataset.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


print("Preprocessing utilities defined.")

## 3. Model Builders

In [ ]:
def load_checkpoint_safely(path, device):
    """Load a state dict from a checkpoint file.

    Handles both raw state dicts and wrapped dicts that store the
    weights under a key such as 'state_dict' or 'model'.
    """
    checkpoint = torch.load(path, map_location=device)
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model", "model_state_dict"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
    return checkpoint


def reinitialize_linear_head(module):
    """Reset all Linear layers inside a module using truncated normal init."""
    for child in module.modules():
        if isinstance(child, nn.Linear):
            nn.init.trunc_normal_(child.weight, std=0.02)
            if child.bias is not None:
                nn.init.zeros_(child.bias)


def create_mobilenet_v2(num_classes, dropout=0.3):
    model = models.mobilenet_v2(weights=None)
    pretrained_path = "pretrained_models/mobilenet_v2-7ebf99e0.pth"
    if os.path.exists(pretrained_path):
        state_dict = load_checkpoint_safely(pretrained_path, device)
        model.load_state_dict(state_dict, strict=False)
    model.classifier = nn.Sequential(
        nn.Dropout(dropout), nn.Linear(model.last_channel, num_classes)
    )
    reinitialize_linear_head(model.classifier)
    return model


def create_shufflenet_v2(num_classes, dropout=0.3):
    model = models.shufflenet_v2_x1_0(weights=None)
    pretrained_path = "pretrained_models/shufflenetv2_x1-5666bf0f80.pth"
    if os.path.exists(pretrained_path):
        state_dict = load_checkpoint_safely(pretrained_path, device)
        model.load_state_dict(state_dict, strict=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout), nn.Linear(in_features, num_classes)
    )
    reinitialize_linear_head(model.fc)
    return model


def create_efficientnet_b3(num_classes, dropout=0.4):
    model = timm.create_model(
        "efficientnet_b3",
        pretrained=False,
        num_classes=num_classes,
        drop_rate=dropout,
    )
    pretrained_path = "pretrained_models/efficientnet_b3_rwightman-b3899882.pth"
    if os.path.exists(pretrained_path):
        state_dict = load_checkpoint_safely(pretrained_path, device)
        model.load_state_dict(state_dict, strict=False)
    if hasattr(model, "classifier"):
        reinitialize_linear_head(model.classifier)
    return model


def create_deit_tiny(num_classes, dropout=0.2):
    model = timm.create_model(
        "deit_tiny_patch16_224",
        pretrained=False,
        num_classes=num_classes,
        drop_rate=dropout,
        drop_path_rate=0.1,
    )
    pretrained_path = "pretrained_models/deit_tiny_patch16_224-a1311bcf.pth"
    if os.path.exists(pretrained_path):
        state_dict = load_checkpoint_safely(pretrained_path, device)
        model_dict = model.state_dict()
        filtered = {
            k: v
            for k, v in state_dict.items()
            if k in model_dict and not k.startswith("head")
        }
        model_dict.update(filtered)
        model.load_state_dict(model_dict, strict=False)
    reinitialize_linear_head(model.head)
    return model


def create_swin_tiny(num_classes, dropout=0.2):
    model = timm.create_model(
        "swin_tiny_patch4_window7_224",
        pretrained=False,
        num_classes=num_classes,
        drop_rate=dropout,
        drop_path_rate=0.15,
    )
    pretrained_path = "pretrained_models/swin_tiny_patch4_window7_224.pth"
    if os.path.exists(pretrained_path):
        print(f"Loading pretrained weights from {pretrained_path}")
        state_dict = load_checkpoint_safely(pretrained_path, device)
        model_dict = model.state_dict()
        filtered = {
            k: v
            for k, v in state_dict.items()
            if k in model_dict
            and model_dict[k].shape == v.shape
            and not k.startswith("head")
        }
        missing = [
            k
            for k in model_dict.keys()
            if k not in filtered and not k.startswith("head")
        ]
        print(
            f"Loaded {len(filtered)} matching tensors, skipped {len(missing)} tensors"
        )
        model_dict.update(filtered)
        model.load_state_dict(model_dict, strict=False)
    reinitialize_linear_head(model.head)
    return model


print("Model builders defined.")

## 4. Model Configuration

Set use_sobel per model and batch sizes here. Sobel preprocessing is disabled by
default for the cotton dataset because colour is a meaningful diagnostic cue for
plant disease. Enable it per model if you want to experiment with edge-only inputs.

In [ ]:
MODEL_CONFIGS = [
    {
        "name": "MobileNetV2",
        "builder": create_mobilenet_v2,
        "type": "cnn",
        "use_sobel": False,
        "batch_size": 16,
    },
    {
        "name": "ShuffleNetV2",
        "builder": create_shufflenet_v2,
        "type": "cnn",
        "use_sobel": False,
        "batch_size": 16,
    },
    {
        "name": "EfficientNet-B3",
        "builder": create_efficientnet_b3,
        "type": "cnn",
        "use_sobel": False,
        "batch_size": 16,
    },
    {
        "name": "DeiT-Tiny",
        "builder": create_deit_tiny,
        "type": "transformer",
        "use_sobel": False,
        "batch_size": 16,
    },
    {
        "name": "Swin-Tiny",
        "builder": create_swin_tiny,
        "type": "transformer",
        "use_sobel": False,
        "batch_size": 16,
    },
]

## 5. Optimizer, Scheduler, and Backbone Freeze Helpers

In [ ]:
def split_parameter_groups(model, model_type):
    """Return (backbone_params, head_params) with correct learning rate groups."""
    if model_type == "transformer":
        head_names = ("head", "fc_norm")
    else:
        head_names = ("classifier", "fc")

    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(key in name for key in head_names):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return backbone_params, head_params


def freeze_backbone(model, model_type, freeze=True):
    """Freeze or unfreeze all backbone parameters while keeping the head trainable."""
    if model_type == "transformer":
        classifier_names = ("head", "fc_norm")
    else:
        classifier_names = ("classifier", "fc")

    for name, param in model.named_parameters():
        if any(key in name for key in classifier_names):
            param.requires_grad = True
        else:
            param.requires_grad = not freeze


def build_optimizer_and_scheduler(
    model, model_type, steps_per_epoch, max_epochs, warmup_epochs
):
    """Build AdamW with layer-wise learning rates and a warmup-cosine schedule."""
    backbone_params, head_params = split_parameter_groups(model, model_type)

    if model_type == "transformer":
        optimizer = optim.AdamW(
            [
                {"params": backbone_params, "lr": 3e-5, "weight_decay": 0.05},
                {"params": head_params, "lr": 2e-4, "weight_decay": 0.02},
            ]
        )
    else:
        optimizer = optim.AdamW(
            [
                {"params": backbone_params, "lr": 3e-4, "weight_decay": 0.01},
                {"params": head_params, "lr": 1e-3, "weight_decay": 0.01},
            ]
        )

    total_steps = max(steps_per_epoch * max_epochs, 1)
    warmup_steps = max(steps_per_epoch * warmup_epochs, 1)

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step + 1) / float(warmup_steps)
        progress = (current_step - warmup_steps) / max(
            total_steps - warmup_steps, 1
        )
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler


print("Optimizer and scheduler helpers defined.")

## 6. Training and Evaluation Functions

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(title)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    return cm


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = correct = total = 0
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        total += targets.size(0)
        correct += (predictions == targets).sum().item()
    return running_loss / max(len(loader), 1), 100.0 * correct / max(total, 1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = correct = total = 0
    all_predictions, all_targets = [], []
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        running_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        total += targets.size(0)
        correct += (predictions == targets).sum().item()
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    return (
        running_loss / max(len(loader), 1),
        100.0 * correct / max(total, 1),
        all_predictions,
        all_targets,
    )


print("Training and evaluation functions defined.")

## 7. K-Fold Training Loop

In [ ]:
def train_kfold(
    model_config,
    dataset,
    n_splits=5,
    max_epochs=60,
    patience=12,
    warmup_epochs=6,
    freeze_backbone_epochs=3,
):
    """Train a single model with stratified k-fold cross-validation.

    Saves the best-fold checkpoint to SAVED_MODELS_DIR with backbone and
    head state dicts stored separately so the fusion notebook can load
    just the backbone without the classifier head.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    labels = np.array([lbl for _, lbl in dataset.samples])

    fold_results = []
    all_predictions = []
    all_targets_out = []
    best_model_state = None
    best_model_fold = None
    best_model_acc = -1.0

    train_transform, val_transform = build_transforms(
        use_sobel=model_config["use_sobel"],
        is_transformer=model_config["type"] == "transformer",
    )

    print("\n" + "=" * 70)
    print(f"Training: {model_config['name']}")
    print("=" * 70)
    print(
        f"use_sobel={model_config['use_sobel']}  type={model_config['type']}"
    )

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(np.zeros(len(labels)), labels), start=1
    ):
        if device.type == "mps":
            torch.mps.empty_cache()
        elif device.type == "cuda":
            torch.cuda.empty_cache()

        train_ds = TransformSubset(dataset, train_idx, train_transform)
        val_ds = TransformSubset(dataset, val_idx, val_transform)

        train_loader = DataLoader(
            train_ds,
            batch_size=model_config["batch_size"],
            shuffle=True,
            num_workers=4,
            pin_memory=True,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=model_config["batch_size"] * 2,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

        model = model_config["builder"](num_classes=NUM_CLASSES).to(device)
        optimizer, scheduler = build_optimizer_and_scheduler(
            model,
            model_config["type"],
            steps_per_epoch=len(train_loader),
            max_epochs=max_epochs,
            warmup_epochs=warmup_epochs,
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

        if model_config["type"] == "transformer" and freeze_backbone_epochs > 0:
            freeze_backbone(model, model_config["type"], freeze=True)

        best_val_acc = -1.0
        best_state = None
        best_epoch = 0
        no_improve = 0
        global_step = 0

        print("-" * 70)
        print(f"Fold {fold}/{n_splits}")
        print("-" * 70)

        for epoch in range(1, max_epochs + 1):
            if (
                model_config["type"] == "transformer"
                and epoch == freeze_backbone_epochs + 1
            ):
                freeze_backbone(model, model_config["type"], freeze=False)

            train_loss, train_acc = train_one_epoch(
                model, train_loader, optimizer, criterion
            )
            val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

            for _ in range(len(train_loader)):
                scheduler.step()
                global_step += 1

            if epoch == 1 or epoch % 10 == 0 or val_acc > best_val_acc:
                lrs = [round(g["lr"], 7) for g in optimizer.param_groups]
                print(
                    f"Epoch {epoch:03d}  "
                    f"train_loss={train_loss:.4f}  train_acc={train_acc:.2f}%  "
                    f"val_loss={val_loss:.4f}  val_acc={val_acc:.2f}%  "
                    f"lrs={lrs}"
                )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        model.load_state_dict(best_state)
        _, fold_acc, fold_preds, fold_targets = evaluate(
            model, val_loader, criterion
        )
        all_predictions.extend(fold_preds)
        all_targets_out.extend(fold_targets)

        fold_results.append(
            {"fold": fold, "accuracy": fold_acc, "best_epoch": best_epoch}
        )

        if fold_acc > best_model_acc:
            best_model_acc = fold_acc
            best_model_fold = fold
            best_model_state = copy.deepcopy(best_state)

        print(f"Fold {fold} best val accuracy: {fold_acc:.2f}%")

    mean_acc = float(np.mean([r["accuracy"] for r in fold_results]))
    std_acc = float(np.std([r["accuracy"] for r in fold_results]))

    cm_path = os.path.join(
        OUTPUT_DIR_INDIVIDUALS,
        "confusion_matrices",
        f"{model_config['name'].lower().replace('-', '_')}_kfold.png",
    )
    plot_confusion_matrix(
        all_targets_out,
        all_predictions,
        CLASS_NAMES,
        f"{model_config['name']} K-Fold CV",
        cm_path,
    )

    print("\nClassification report:")
    print(
        classification_report(
            all_targets_out,
            all_predictions,
            target_names=CLASS_NAMES,
            digits=4,
        )
    )

    # Determine head key prefix so backbone and head can be stored separately
    if model_config["type"] == "transformer":
        head_prefix = "head."
    else:
        head_prefix = "fc."

    checkpoint_name = (
        f"{model_config['name'].lower().replace('-', '_')}"
        f"_best_fold{best_model_fold}.pth"
    )
    checkpoint_path = os.path.join(SAVED_MODELS_DIR, checkpoint_name)

    torch.save(
        {
            "model_name": model_config["name"],
            "model_type": model_config["type"],
            "use_sobel": model_config["use_sobel"],
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "best_fold": best_model_fold,
            "best_fold_accuracy": best_model_acc,
            "mean_accuracy": mean_acc,
            "std_accuracy": std_acc,
            "state_dict": best_model_state,
            "backbone_state_dict": {
                k: v
                for k, v in best_model_state.items()
                if not k.startswith(head_prefix)
            },
            "head_state_dict": {
                k: v
                for k, v in best_model_state.items()
                if k.startswith(head_prefix)
            },
        },
        checkpoint_path,
    )
    print(f"Saved checkpoint: {checkpoint_path}")

    return {
        "model_name": model_config["name"],
        "mean_accuracy": mean_acc,
        "std_accuracy": std_acc,
        "fold_results": fold_results,
        "use_sobel": model_config["use_sobel"],
        "model_type": model_config["type"],
        "best_model_path": checkpoint_path,
        "best_model_fold": best_model_fold,
        "best_model_accuracy": best_model_acc,
    }


print("K-fold training loop defined.")

## 8. Run All Individual Models

In [ ]:
set_seed()

N_SPLITS = 5
MAX_EPOCHS = 60
PATIENCE = 12
WARMUP_EPOCHS = 6
FREEZE_BACKBONE_EPOCHS = 3

individual_results = []
start_time = time.time()

for config in MODEL_CONFIGS:
    try:
        result = train_kfold(
            config,
            full_dataset,
            n_splits=N_SPLITS,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            warmup_epochs=WARMUP_EPOCHS,
            freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS,
        )
        individual_results.append(result)

        # Record checkpoint paths for the fusion backbones
        if config["name"] == "DeiT-Tiny":
            DEIT_CKPT = result["best_model_path"]
        if config["name"] == "ShuffleNetV2":
            SHUFFLENET_CKPT = result["best_model_path"]

    except Exception as exc:
        print(f"Error training {config['name']}: {exc}")

elapsed = (time.time() - start_time) / 60.0
print(f"\nAll individual models completed in {elapsed:.1f} minutes")

## 9. Individual Model Summary

In [ ]:
print("\n" + "=" * 70)
print("INDIVIDUAL MODEL RESULTS SUMMARY")
print("=" * 70)
print(f"{'Model':<20} {'Mean Acc':>10} {'Std':>7} {'Best Fold Acc':>14}")
print("-" * 55)
for r in sorted(individual_results, key=lambda x: x["mean_accuracy"], reverse=True):
    print(
        f"{r['model_name']:<20} "
        f"{r['mean_accuracy']:>9.2f}%  "
        f"+-{r['std_accuracy']:>4.2f}%  "
        f"{r['best_model_accuracy']:>13.2f}%"
    )
print("=" * 70)

# Save text summary
summary_path = os.path.join(OUTPUT_DIR_INDIVIDUALS, "kfold_summary.txt")
with open(summary_path, "w") as f:
    f.write("COTTON DISEASE CLASSIFICATION - INDIVIDUAL MODELS\n")
    f.write("K-FOLD CROSS-VALIDATION RESULTS\n")
    f.write("=" * 70 + "\n")
    for r in sorted(
        individual_results, key=lambda x: x["mean_accuracy"], reverse=True
    ):
        f.write(
            f"{r['model_name']}: {r['mean_accuracy']:.2f}% "
            f"+- {r['std_accuracy']:.2f}%  "
            f"use_sobel={r['use_sobel']}  "
            f"type={r['model_type']}  "
            f"best_path={r['best_model_path']}\n"
        )
        for fr in r["fold_results"]:
            f.write(
                f"  Fold {fr['fold']}: {fr['accuracy']:.2f}%  "
                f"(best_epoch={fr['best_epoch']})\n"
            )
        f.write("\n")

print(f"Summary saved to {summary_path}")
print(f"\nDeiT-Tiny checkpoint  : {DEIT_CKPT}")
print(f"ShuffleNetV2 checkpoint: {SHUFFLENET_CKPT}")

---
# Part 2 - Feature Fusion

The DeiT-Tiny and ShuffleNetV2 backbones trained above are loaded and frozen.
Lightweight fusion heads are then trained on top of the frozen feature vectors.

Five fusion strategies are evaluated:
- concat         : concatenate both feature vectors, pass through a deep MLP
- weighted_sum   : project to a common dimension, sum with learnable scalar weights
- bilinear       : Hadamard product followed by signed-sqrt normalisation
- cross_attention: bidirectional cross-attention between the two feature vectors
- gated          : per-feature sigmoid gate controls how much each backbone contributes

## 10. Dual-Transform Dataset for Fusion

In [ ]:
# Standard ImageNet normalisation used by both branches
_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)

# DeiT branch transform (matches how DeiT-Tiny was trained above)
transform_deit_train = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        _normalize,
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    ]
)

transform_deit_val = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        _normalize,
    ]
)

# ShuffleNetV2 branch transform (matches how ShuffleNetV2 was trained above)
transform_shuffle_train = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        _normalize,
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    ]
)

transform_shuffle_val = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        _normalize,
    ]
)


class DualTransformSubset(Dataset):
    """Return two differently-transformed views of the same image and its label.

    Used to feed each backbone branch its own augmentation pipeline
    without loading the image twice from disk.
    """

    def __init__(self, subset: Subset, transform_a, transform_b):
        self.subset = subset
        self.transform_a = transform_a
        self.transform_b = transform_b
        self.base_dataset = subset.dataset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        real_idx = self.subset.indices[idx]
        path, label = self.base_dataset.samples[real_idx]
        img = Image.open(path).convert("RGB")
        return self.transform_a(img), self.transform_b(img), label


def make_dual_loaders(dataset, train_idx, val_idx, batch_size=16):
    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    train_ds = DualTransformSubset(
        train_sub, transform_deit_train, transform_shuffle_train
    )
    val_ds = DualTransformSubset(
        val_sub, transform_deit_val, transform_shuffle_val
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
    return train_loader, val_loader


print("Dual-transform dataset and data loaders defined.")

## 11. Load Frozen Backbones

In [ ]:
def load_deit_tiny_backbone(ckpt_path: str):
    """Load a DeiT-Tiny checkpoint and return the frozen backbone and its feature dim."""
    model = timm.create_model(
        "deit_tiny_patch16_224", pretrained=False, num_classes=NUM_CLASSES
    )
    feat_dim = model.head.in_features  # 192

    ckpt = torch.load(ckpt_path, map_location="cpu")
    assert isinstance(ckpt, dict), f"Expected dict checkpoint, got {type(ckpt)}"
    assert "state_dict" in ckpt, (
        f"Key 'state_dict' not found. Available keys: {list(ckpt.keys())}"
    )

    state = ckpt["state_dict"]
    missing, unexpected = model.load_state_dict(state, strict=True)
    assert len(missing) == 0, f"Missing keys: {missing}"
    assert len(unexpected) == 0, f"Unexpected keys: {unexpected}"

    norm = sum(p.norm().item() for p in model.parameters())
    print(f"DeiT-Tiny loaded from {ckpt_path}")
    print(f"  best_fold={ckpt.get('best_fold')}  fold_acc={ckpt.get('best_fold_accuracy'):.2f}%")
    print(f"  weight norm: {norm:.2f}")
    assert norm > 10, f"Weight norm suspiciously low ({norm:.2f})"

    for p in model.parameters():
        p.requires_grad_(False)
    model.head = nn.Identity()
    model.eval()
    return model, feat_dim


def load_shufflenet_backbone(ckpt_path: str):
    """Load a ShuffleNetV2 checkpoint and return the frozen backbone and its feature dim."""
    model = models.shufflenet_v2_x1_0(weights=None, num_classes=NUM_CLASSES)
    feat_dim = model.fc.in_features  # 1024

    ckpt = torch.load(ckpt_path, map_location="cpu")
    assert isinstance(ckpt, dict), f"Expected dict checkpoint, got {type(ckpt)}"
    assert "state_dict" in ckpt, (
        f"Key 'state_dict' not found. Available keys: {list(ckpt.keys())}"
    )

    state = ckpt["state_dict"]
    backbone_state = {k: v for k, v in state.items() if not k.startswith("fc.")}
    missing, unexpected = model.load_state_dict(backbone_state, strict=False)
    assert all(k.startswith("fc.") for k in missing), (
        f"Unexpected missing keys: {missing}"
    )
    assert len(unexpected) == 0, f"Unexpected keys: {unexpected}"

    norm = sum(p.norm().item() for p in model.parameters())
    print(f"ShuffleNetV2 loaded from {ckpt_path}")
    print(f"  best_fold={ckpt.get('best_fold')}  fold_acc={ckpt.get('best_fold_accuracy'):.2f}%")
    print(f"  weight norm: {norm:.2f}")
    assert norm > 10, f"Weight norm suspiciously low ({norm:.2f})"

    for p in model.parameters():
        p.requires_grad_(False)
    model.fc = nn.Identity()
    model.eval()
    return model, feat_dim


print("Loading backbones...")
deit_backbone, DEIT_DIM = load_deit_tiny_backbone(DEIT_CKPT)
shuffle_backbone, SHUFFLE_DIM = load_shufflenet_backbone(SHUFFLENET_CKPT)

deit_backbone = deit_backbone.to(device)
shuffle_backbone = shuffle_backbone.to(device)

print(f"\nFeature dims  DeiT: {DEIT_DIM}  ShuffleNetV2: {SHUFFLE_DIM}")
print("Both backbones are frozen.")

## 12. Baseline Accuracy for Comparison

In [ ]:
# Pull the mean cross-validation accuracy from the individual model results
# so the fusion summary can show deltas against these baselines.

def get_mean_acc(name):
    for r in individual_results:
        if r["model_name"] == name:
            return r["mean_accuracy"]
    return None


BASELINE_DEIT = get_mean_acc("DeiT-Tiny")
BASELINE_SHUFFLENET = get_mean_acc("ShuffleNetV2")

print(f"DeiT-Tiny baseline (CV mean)   : {BASELINE_DEIT:.2f}%")
print(f"ShuffleNetV2 baseline (CV mean): {BASELINE_SHUFFLENET:.2f}%")

## 13. Feature Sanity Check

In [ ]:
print("Running feature sanity check...")

all_idx = list(range(len(full_dataset)))
_, val_idx_check = next(
    iter(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(
            np.zeros(len(all_idx)),
            [full_dataset.targets[i] for i in all_idx],
        )
    )
)
train_idx_check = list(set(all_idx) - set(val_idx_check))

check_loader, _ = make_dual_loaders(
    full_dataset, train_idx_check, val_idx_check, batch_size=32
)

deit_feats_all, shuffle_feats_all, labels_all = [], [], []

with torch.no_grad():
    for deit_imgs, shuffle_imgs, lbls in check_loader:
        deit_imgs = deit_imgs.to(device)
        shuffle_imgs = shuffle_imgs.to(device)
        deit_feats_all.append(deit_backbone(deit_imgs).cpu())
        shuffle_feats_all.append(shuffle_backbone(shuffle_imgs).cpu())
        labels_all.extend(lbls.tolist())
        if len(labels_all) >= 100:
            break

deit_feats_all = torch.cat(deit_feats_all)
shuffle_feats_all = torch.cat(shuffle_feats_all)

print(f"DeiT features    shape: {deit_feats_all.shape}")
print(
    f"  Mean={deit_feats_all.mean():.4f}  Std={deit_feats_all.std():.4f}  "
    f"Min={deit_feats_all.min():.4f}  Max={deit_feats_all.max():.4f}"
)
print(f"ShuffleNet features shape: {shuffle_feats_all.shape}")
print(
    f"  Mean={shuffle_feats_all.mean():.4f}  Std={shuffle_feats_all.std():.4f}  "
    f"Min={shuffle_feats_all.min():.4f}  Max={shuffle_feats_all.max():.4f}"
)

labels_arr = np.array(labels_all[: len(deit_feats_all)])
global_mean = deit_feats_all.mean(0)
print("\nPer-class feature means (DeiT) -- L2 distance from global mean:")
for c, name in enumerate(CLASS_NAMES):
    idx = np.where(labels_arr == c)[0]
    if len(idx) == 0:
        continue
    cmean = deit_feats_all[idx].mean(0)
    dist = (cmean - global_mean).norm().item()
    print(f"  {name:<20}: {dist:.4f}  (n={len(idx)})")

global_mean_s = shuffle_feats_all.mean(0)
print("\nPer-class feature means (ShuffleNetV2) -- L2 distance from global mean:")
for c, name in enumerate(CLASS_NAMES):
    idx = np.where(labels_arr == c)[0]
    if len(idx) == 0:
        continue
    cmean = shuffle_feats_all[idx].mean(0)
    dist = (cmean - global_mean_s).norm().item()
    print(f"  {name:<20}: {dist:.4f}  (n={len(idx)})")

print("\nIf Std >> 0 and class distances differ, features are discriminative.")
print("If Std is near 0 or all distances are equal, the checkpoint load may have failed.")

## 14. Fusion Head Architectures

In [ ]:
class ConcatFusion(nn.Module):
    """Concatenate both feature vectors and pass through a deep MLP."""

    def __init__(self, d1, d2, num_classes, dropout=0.4):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d1 + d2, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, f1, f2):
        return self.head(torch.cat([f1, f2], dim=-1))


class WeightedSumFusion(nn.Module):
    """Project both features to a shared dimension, sum with learnable scalar weights."""

    def __init__(self, d1, d2, num_classes, embed_dim=256, dropout=0.4):
        super().__init__()
        self.proj1 = nn.Linear(d1, embed_dim)
        self.proj2 = nn.Linear(d2, embed_dim)
        self.w = nn.Parameter(torch.ones(2) / 2)
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, f1, f2):
        w = F.softmax(self.w, dim=0)
        return self.head(w[0] * self.proj1(f1) + w[1] * self.proj2(f2))


class BilinearFusion(nn.Module):
    """Hadamard product of projected features, followed by signed-sqrt normalisation."""

    def __init__(self, d1, d2, num_classes, embed_dim=256, dropout=0.4):
        super().__init__()
        self.proj1 = nn.Sequential(nn.Linear(d1, embed_dim), nn.ReLU())
        self.proj2 = nn.Sequential(nn.Linear(d2, embed_dim), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, f1, f2):
        p1 = self.proj1(f1)
        p2 = self.proj2(f2)
        x = p1 * p2
        x = torch.sign(x) * torch.sqrt(x.abs() + 1e-8)
        x = F.normalize(x, p=2, dim=-1)
        return self.head(x)


class CrossAttentionFusion(nn.Module):
    """Bidirectional cross-attention so each backbone attends to the other."""

    def __init__(self, d1, d2, num_classes, embed_dim=256, num_heads=4, dropout=0.4):
        super().__init__()
        self.proj1 = nn.Linear(d1, embed_dim)
        self.proj2 = nn.Linear(d2, embed_dim)
        self.attn_1to2 = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.attn_2to1 = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, 2), nn.Softmax(dim=-1)
        )
        self.head = nn.Sequential(
            nn.Linear(embed_dim * 2, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, f1, f2):
        p1_raw = self.proj1(f1)
        p2_raw = self.proj2(f2)
        p1 = p1_raw.unsqueeze(1)
        p2 = p2_raw.unsqueeze(1)
        o12, _ = self.attn_1to2(p1, p2, p2)
        o21, _ = self.attn_2to1(p2, p1, p1)
        o12 = self.norm1(o12 + p1).squeeze(1)
        o21 = self.norm2(o21 + p2).squeeze(1)
        combined_raw = torch.cat([p1_raw, p2_raw], dim=-1)
        combined_attn = torch.cat([o12, o21], dim=-1)
        weights = self.gate(combined_raw)
        final = weights[:, 0:1] * combined_raw + weights[:, 1:2] * combined_attn
        return self.head(final)


class GatedFusion(nn.Module):
    """Learned per-feature sigmoid gate controls how much each backbone contributes."""

    def __init__(self, d1, d2, num_classes, embed_dim=256, dropout=0.4):
        super().__init__()
        self.proj1 = nn.Sequential(
            nn.Linear(d1, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.proj2 = nn.Sequential(
            nn.Linear(d2, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid(),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.LayerNorm(embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )
        self.temp = nn.Parameter(torch.ones(1) * 0.7)

    def forward(self, f1, f2):
        p1 = self.proj1(f1)
        p2 = self.proj2(f2)
        g_raw = self.gate[0](torch.cat([p1, p2], dim=-1))
        g = torch.sigmoid(g_raw / self.temp)
        for layer in self.gate[1:]:
            g = layer(g)
        x = g * p1 + (1 - g) * p2 + p1 + p2
        return self.head(x)


def build_fusion_head(strategy, d1, d2, num_classes, embed_dim=256, dropout=0.4):
    heads = {
        "concat": ConcatFusion(d1, d2, num_classes, dropout),
        "weighted_sum": WeightedSumFusion(d1, d2, num_classes, embed_dim, dropout),
        "bilinear": BilinearFusion(d1, d2, num_classes, embed_dim, dropout),
        "cross_attention": CrossAttentionFusion(
            d1, d2, num_classes, embed_dim, 4, dropout
        ),
        "gated": GatedFusion(d1, d2, num_classes, embed_dim, dropout),
    }
    assert strategy in heads, (
        f"Unknown strategy '{strategy}'. Available: {list(heads.keys())}"
    )
    return heads[strategy]


FUSION_STRATEGIES = ["concat", "weighted_sum", "bilinear", "cross_attention", "gated"]

descs = {
    "concat": "Concatenate then deep MLP",
    "weighted_sum": "Projection + learnable scalar weights",
    "bilinear": "Hadamard product + signed-sqrt normalisation",
    "cross_attention": "Bidirectional cross-attention",
    "gated": "Per-feature sigmoid gate",
}

print(f"{'Strategy':<22} {'Params':>10}  Description")
print("-" * 70)
for s in FUSION_STRATEGIES:
    h = build_fusion_head(s, DEIT_DIM, SHUFFLE_DIM, NUM_CLASSES)
    n = sum(p.numel() for p in h.parameters())
    print(f"{s:<22} {n:>10,}  {descs[s]}")

## 15. Fusion Training and Evaluation Functions

In [ ]:
def fusion_train_epoch(fusion_head, optimizer, criterion, loader):
    """One epoch of fusion head training with frozen backbone feature extraction."""
    fusion_head.train()
    total_loss = correct = total = 0
    for deit_imgs, shuffle_imgs, lbls in loader:
        deit_imgs = deit_imgs.to(device)
        shuffle_imgs = shuffle_imgs.to(device)
        lbls = lbls.to(device)
        with torch.no_grad():
            f1 = deit_backbone(deit_imgs)
            f2 = shuffle_backbone(shuffle_imgs)
        optimizer.zero_grad()
        logits = fusion_head(f1, f2)
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_head.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == lbls).sum().item()
        total += lbls.size(0)
    return total_loss / len(loader), 100 * correct / total


@torch.no_grad()
def fusion_eval_epoch(fusion_head, criterion, loader):
    """Evaluate a fusion head on a data loader."""
    fusion_head.eval()
    total_loss = correct = total = 0
    all_preds, all_labels_out = [], []
    for deit_imgs, shuffle_imgs, lbls in loader:
        deit_imgs = deit_imgs.to(device)
        shuffle_imgs = shuffle_imgs.to(device)
        lbls = lbls.to(device)
        f1 = deit_backbone(deit_imgs)
        f2 = shuffle_backbone(shuffle_imgs)
        logits = fusion_head(f1, f2)
        loss = criterion(logits, lbls)
        total_loss += loss.item()
        preds = logits.argmax(1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels_out.extend(lbls.cpu().tolist())
    return total_loss / len(loader), 100 * correct / total, all_preds, all_labels_out


def plot_training_history(history, title, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{title} - Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Val")
    axes[1].set_title(f"{title} - Accuracy (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


print("Fusion training and evaluation functions defined.")

## 16. K-Fold Fusion Ablation

In [ ]:
def run_fusion_kfold(
    strategy,
    dataset,
    n_splits=5,
    max_epochs=80,
    patience=20,
    embed_dim=256,
    dropout=0.4,
    lr=3e-4,
    batch_size=16,
):
    """Train a single fusion strategy with k-fold cross-validation."""
    print(f"\n{'='*65}")
    print(f"  FUSION STRATEGY: {strategy.upper()}")
    print(f"{'='*65}")

    targets = [lbl for _, lbl in dataset.samples]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    fold_results, all_preds_cv, all_labels_cv = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(
        skf.split(np.zeros(len(targets)), targets)
    ):
        print(f"\n  Fold {fold+1}/{n_splits}")
        tr_loader, va_loader = make_dual_loaders(
            dataset, tr_idx, va_idx, batch_size
        )

        head = build_fusion_head(
            strategy, DEIT_DIM, SHUFFLE_DIM, NUM_CLASSES, embed_dim, dropout
        ).to(device)

        optimizer = torch.optim.AdamW(
            head.parameters(), lr=lr, weight_decay=1e-2
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-6
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

        history = {
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
        }
        best_acc = 0.0
        best_state = None
        no_improve = 0
        best_epoch = 0

        for epoch in range(max_epochs):
            tl, ta = fusion_train_epoch(head, optimizer, criterion, tr_loader)
            vl, va, vp, vl2 = fusion_eval_epoch(head, criterion, va_loader)
            scheduler.step()

            history["train_loss"].append(tl)
            history["train_acc"].append(ta)
            history["val_loss"].append(vl)
            history["val_acc"].append(va)

            if va > best_acc:
                best_acc = va
                best_state = {k: v.clone() for k, v in head.state_dict().items()}
                no_improve = 0
                best_epoch = epoch + 1
            else:
                no_improve += 1

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(
                    f"    Ep {epoch+1:3d}  "
                    f"train {ta:.2f}%  val {va:.2f}%  best {best_acc:.2f}%"
                )

            if no_improve >= patience:
                print(f"    Early stop at epoch {epoch+1}")
                break

        head.load_state_dict(best_state)
        _, fold_acc, fp, fl = fusion_eval_epoch(head, criterion, va_loader)
        all_preds_cv.extend(fp)
        all_labels_cv.extend(fl)

        fold_results.append(
            {
                "fold": fold + 1,
                "accuracy": fold_acc,
                "best_epoch": best_epoch,
                "history": history,
            }
        )
        print(f"  Fold {fold+1} best: {fold_acc:.2f}%  (epoch {best_epoch})")

        plot_training_history(
            history,
            f"{strategy} Fold {fold+1}",
            os.path.join(
                OUTPUT_DIR_FUSION,
                "visualizations",
                f"{strategy}_fold{fold+1}_history.png",
            ),
        )

    accs = [r["accuracy"] for r in fold_results]
    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs))
    delta_d = mean_acc - BASELINE_DEIT
    delta_s = mean_acc - BASELINE_SHUFFLENET

    print(f"\n  {strategy.upper()} -- {mean_acc:.2f}% +- {std_acc:.2f}%")
    for r in fold_results:
        print(
            f"    Fold {r['fold']}: {r['accuracy']:.2f}%  (best_epoch={r['best_epoch']})"
        )
    print(f"  Delta vs DeiT-Tiny   : {delta_d:+.2f}%")
    print(f"  Delta vs ShuffleNetV2: {delta_s:+.2f}%")

    plot_confusion_matrix(
        all_labels_cv,
        all_preds_cv,
        CLASS_NAMES,
        f"{strategy} ({mean_acc:.2f}% +- {std_acc:.2f}%)",
        os.path.join(
            OUTPUT_DIR_FUSION,
            "confusion_matrices",
            f"{strategy}_cm.png",
        ),
    )

    print(
        classification_report(
            all_labels_cv,
            all_preds_cv,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )
    )

    return {
        "strategy": strategy,
        "mean_accuracy": mean_acc,
        "std_accuracy": std_acc,
        "fold_results": fold_results,
        "all_preds": all_preds_cv,
        "all_labels": all_labels_cv,
        "delta_deit": delta_d,
        "delta_shufflenet": delta_s,
    }


print("Fusion k-fold function defined.")

## 17. Run All Fusion Strategies

In [ ]:
FUSION_CFG = dict(
    n_splits=5,
    max_epochs=80,
    patience=20,
    embed_dim=256,
    dropout=0.4,
    lr=3e-4,
    batch_size=16,
)

fusion_results = {}
for strategy in FUSION_STRATEGIES:
    fusion_results[strategy] = run_fusion_kfold(
        strategy=strategy, dataset=full_dataset, **FUSION_CFG
    )

print("\n" + "=" * 65)
print("ALL FUSION RUNS COMPLETE")
print("=" * 65)

## 18. Fusion Results Summary and Plots

In [ ]:
print("\n" + "=" * 75)
print("FEATURE FUSION ABLATION -- SUMMARY")
print("=" * 75)
print(
    f"{'Model / Strategy':<24} {'Mean Acc':>10} {'+-Std':>7} "
    f"{'D-DeiT':>9} {'D-Shuffle':>11}"
)
print("-" * 75)
print(
    f"{'[DeiT-Tiny]':<24} {BASELINE_DEIT:>9.2f}%  "
    f"{'--':>6}  {'--':>8}  {'--':>10}"
)
print(
    f"{'[ShuffleNetV2]':<24} {BASELINE_SHUFFLENET:>9.2f}%  "
    f"{'--':>6}  {'--':>8}  {'--':>10}"
)
print("-" * 75)

rows = []
best_individual = max(BASELINE_DEIT, BASELINE_SHUFFLENET)
for s, r in fusion_results.items():
    rows.append(
        (s, r["mean_accuracy"], r["std_accuracy"], r["delta_deit"], r["delta_shufflenet"])
    )
    flag = " ***" if r["mean_accuracy"] > best_individual else ""
    print(
        f"{s:<24} {r['mean_accuracy']:>9.2f}%  "
        f"+-{r['std_accuracy']:>5.2f}%  "
        f"{r['delta_deit']:>+7.2f}%  "
        f"{r['delta_shufflenet']:>+9.2f}%{flag}"
    )
print("=" * 75)
print("*** marks strategies that outperform both individual baselines")

# Bar chart
strategies = [r[0] for r in rows]
means = [r[1] for r in rows]
stds = [r[2] for r in rows]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    np.arange(len(strategies)),
    means,
    yerr=stds,
    capsize=5,
    color=colors,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
)
ax.axhline(
    BASELINE_DEIT,
    color="steelblue",
    linestyle="--",
    linewidth=1.5,
    label=f"DeiT-Tiny baseline ({BASELINE_DEIT:.2f}%)",
)
ax.axhline(
    BASELINE_SHUFFLENET,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f"ShuffleNetV2 baseline ({BASELINE_SHUFFLENET:.2f}%)",
)
for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + s + 0.2,
        f"{m:.2f}%",
        ha="center",
        va="bottom",
        fontsize=8,
        fontweight="bold",
    )
ax.set_xticks(np.arange(len(strategies)))
ax.set_xticklabels([s.replace("_", "\n") for s in strategies], fontsize=9)
ax.set_ylabel("5-Fold CV Accuracy (%)")
ax.set_title(
    "Cotton Disease - Feature Fusion Ablation (DeiT-Tiny + ShuffleNetV2)",
    fontweight="bold",
)
ax.legend(fontsize=9)
ax.set_ylim(
    max(0, min(means + [BASELINE_SHUFFLENET]) - 5),
    min(100, max(means + [BASELINE_DEIT]) + 6),
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
bar_path = os.path.join(
    OUTPUT_DIR_FUSION, "visualizations", "fusion_ablation_bar.png"
)
plt.savefig(bar_path, dpi=200, bbox_inches="tight")
plt.show()

# Box plot
fig2, ax2 = plt.subplots(figsize=(12, 5))
fold_accs = {
    s: [r["accuracy"] for r in fusion_results[s]["fold_results"]]
    for s in FUSION_STRATEGIES
}
bp = ax2.boxplot(
    fold_accs.values(),
    labels=FUSION_STRATEGIES,
    patch_artist=True,
    widths=0.5,
)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax2.axhline(
    BASELINE_DEIT,
    color="steelblue",
    linestyle="--",
    linewidth=1.5,
    label=f"DeiT-Tiny ({BASELINE_DEIT:.2f}%)",
)
ax2.axhline(
    BASELINE_SHUFFLENET,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f"ShuffleNetV2 ({BASELINE_SHUFFLENET:.2f}%)",
)
ax2.set_title(
    "Fold Accuracy Distribution by Fusion Strategy", fontweight="bold"
)
ax2.set_xlabel("Fusion Strategy")
ax2.set_ylabel("Fold Accuracy (%)")
ax2.set_xticklabels(
    [s.replace("_", "\n") for s in FUSION_STRATEGIES], fontsize=9
)
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(
        OUTPUT_DIR_FUSION, "visualizations", "fusion_ablation_boxplot.png"
    ),
    dpi=200,
    bbox_inches="tight",
)
plt.show()

best_s = max(fusion_results, key=lambda s: fusion_results[s]["mean_accuracy"])
best_r = fusion_results[best_s]
print(
    f"\nBest fusion strategy: {best_s.upper()}  "
    f"{best_r['mean_accuracy']:.2f}% +- {best_r['std_accuracy']:.2f}%"
)

## 19. Per-Fold Accuracy Breakdown and Report

In [ ]:
print("Per-fold accuracy breakdown:")
hdr = f"{'Fold':>5}" + "".join(f" {s:>20}" for s in FUSION_STRATEGIES)
print(hdr)
print("-" * len(hdr))
for fi in range(1, N_SPLITS + 1):
    row = f"{fi:>5}"
    for s in FUSION_STRATEGIES:
        row += f" {fusion_results[s]['fold_results'][fi-1]['accuracy']:>19.2f}%"
    print(row)
print("-" * len(hdr))
row = f"{'Mean':>5}"
for s in FUSION_STRATEGIES:
    row += f" {fusion_results[s]['mean_accuracy']:>19.2f}%"
print(row)
row = f"{'Std':>5}"
for s in FUSION_STRATEGIES:
    row += f" {fusion_results[s]['std_accuracy']:>19.2f}%"
print(row)

# Save text report
report_path = os.path.join(OUTPUT_DIR_FUSION, "fusion_ablation_report.txt")
with open(report_path, "w") as f:
    f.write("COTTON DISEASE CLASSIFICATION - FEATURE FUSION ABLATION\n")
    f.write("Backbones: DeiT-Tiny + ShuffleNetV2 (frozen)\n")
    f.write("=" * 75 + "\n\n")
    f.write(f"DeiT-Tiny baseline   : {BASELINE_DEIT:.2f}%\n")
    f.write(f"ShuffleNetV2 baseline: {BASELINE_SHUFFLENET:.2f}%\n\n")
    for s, r in fusion_results.items():
        f.write(
            f"{s}: {r['mean_accuracy']:.2f}% +- {r['std_accuracy']:.2f}%  "
            f"D_DeiT={r['delta_deit']:+.2f}%  "
            f"D_Shuffle={r['delta_shufflenet']:+.2f}%\n"
        )
        for fr in r["fold_results"]:
            f.write(
                f"  Fold {fr['fold']}: {fr['accuracy']:.2f}%  "
                f"(best_epoch={fr['best_epoch']})\n"
            )
        f.write(
            classification_report(
                r["all_labels"],
                r["all_preds"],
                target_names=CLASS_NAMES,
                digits=4,
                zero_division=0,
            )
        )
        f.write("\n")
    f.write(
        f"\nBest: {best_s}  "
        f"{best_r['mean_accuracy']:.2f}% +- {best_r['std_accuracy']:.2f}%\n"
    )

print(f"\nFusion report saved to {report_path}")